# Step 3 — Week 1: Embeddings & Vector Databases

## Goals
- Understand what embeddings are and how similarity search works.
- Compare embedding models by quality and speed.
- Store and query vectors in ChromaDB using local Ollama (`llama3.2`).

## Prerequisites
- Ollama installed and running locally: https://ollama.com
- Model pulled: `ollama pull llama3.2`
- ChromaDB and sentence-transformers installed (see Setup cell below)

## Setup — Install Dependencies

In [ ]:
%pip install chromadb sentence-transformers ollama requests -q

---
## Part 1 — What is an Embedding?

An **embedding** is a dense numerical vector that captures the *semantic meaning* of text.
Similar sentences produce vectors that are geometrically close to each other.

```
Text: "The cat sat on the mat"
  → Embedding model →
Vector: [0.23, -0.11, 0.87, ..., 0.04]  (768 or 1536 dimensions)
```

This is the foundation of semantic search, RAG, clustering, and recommendations.

### 1a) Generate Embeddings with Ollama (`llama3.2`)
Ollama's `/api/embeddings` endpoint returns a vector for any text using your local model.
No internet or API key required.

In [1]:
import requests
import json

OLLAMA_URL = "http://localhost:11434"
EMBED_MODEL = "llama3.2"

def embed_ollama(text: str, model: str = EMBED_MODEL) -> list[float]:
    response = requests.post(
        f"{OLLAMA_URL}/api/embeddings",
        json={"model": model, "prompt": text},
        timeout=60,
    )
    response.raise_for_status()
    return response.json()["embedding"]

sample_text = "Python is a great programming language for AI."
vec = embed_ollama(sample_text)

print(f"Model: {EMBED_MODEL}")
print(f"Embedding dimensions: {len(vec)}")
print(f"First 8 values: {[round(v, 4) for v in vec[:8]]}")

Model: llama3.2
Embedding dimensions: 3072
First 8 values: [-1.2222, 0.7372, -3.688, -0.0544, 1.387, -0.9112, 0.5083, 1.6807]


### 1b) Why do similar sentences produce similar vectors?
Let's visualise the intuition by comparing raw embeddings of related vs unrelated sentences.

In [2]:
sentences = [
    "Python is a popular language for machine learning.",
    "Machine learning uses Python extensively.",
    "I enjoy hiking in the mountains on weekends.",
    "The Eiffel Tower is located in Paris, France.",
]

embeddings = {s: embed_ollama(s) for s in sentences}
print("Embeddings generated for", len(embeddings), "sentences.")
print("Dimension:", len(list(embeddings.values())[0]))

Embeddings generated for 4 sentences.
Dimension: 3072


---
## Part 2 — Similarity Search

The most common similarity metric for embeddings is **cosine similarity**:

$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}$$

- Value of **1.0** → identical direction (very similar meaning)
- Value of **0.0** → orthogonal (unrelated)
- Value of **-1.0** → opposite meaning

In [3]:
import math

def dot(a: list[float], b: list[float]) -> float:
    return sum(x * y for x, y in zip(a, b))

def magnitude(v: list[float]) -> float:
    return math.sqrt(sum(x ** 2 for x in v))

def cosine_similarity(a: list[float], b: list[float]) -> float:
    mag = magnitude(a) * magnitude(b)
    if mag == 0:
        return 0.0
    return dot(a, b) / mag

reference = sentences[0]
ref_vec = embeddings[reference]

print(f"Reference: \"{reference}\"")
print()
for s in sentences[1:]:
    sim = cosine_similarity(ref_vec, embeddings[s])
    print(f"  [{sim:+.4f}]  {s}")

Reference: "Python is a popular language for machine learning."

  [+0.9468]  Machine learning uses Python extensively.
  [+0.5674]  I enjoy hiking in the mountains on weekends.
  [+0.5934]  The Eiffel Tower is located in Paris, France.


### 2b) Brute-force Nearest Neighbour Search
Given a query, find the most semantically similar document from a small corpus.

In [4]:
def nearest_neighbours(query: str, corpus: dict, top_k: int = 3) -> list[tuple[float, str]]:
    query_vec = embed_ollama(query)
    scores = [
        (cosine_similarity(query_vec, vec), text)
        for text, vec in corpus.items()
    ]
    return sorted(scores, reverse=True)[:top_k]

query = "Which programming language is best for AI?"
results = nearest_neighbours(query, embeddings, top_k=3)

print(f"Query: \"{query}\"")
print()
for rank, (score, text) in enumerate(results, 1):
    print(f"  #{rank} [{score:+.4f}]  {text}")

Query: "Which programming language is best for AI?"

  #1 [+0.6097]  Machine learning uses Python extensively.
  #2 [+0.5947]  Python is a popular language for machine learning.
  #3 [+0.3985]  I enjoy hiking in the mountains on weekends.


---
## Part 3 — Compare Embedding Models: Quality vs Speed

We compare three approaches:

| Model | Type | Speed | Notes |
|---|---|---|---|
| `llama3.2` | Local Ollama LLM | Slower | Full LLM embeddings |
| `nomic-embed-text` | Local Ollama embed-only | Fast | Dedicated embed model |
| `all-MiniLM-L6-v2` | SentenceTransformers (CPU) | Very fast | Lightweight, open source |

In [5]:
import time
from sentence_transformers import SentenceTransformer

BENCHMARK_TEXTS = [
    "Retrieval-Augmented Generation improves LLM accuracy.",
    "ChromaDB is a lightweight vector database.",
    "Cosine similarity measures the angle between vectors.",
    "Large language models hallucinate when lacking grounding.",
    "Embeddings encode semantic meaning in dense vectors.",
]

results_table = {}

# Model 1: llama3.2 via Ollama
start = time.perf_counter()
llama_vecs = [embed_ollama(t, model="llama3.2") for t in BENCHMARK_TEXTS]
llama_time = time.perf_counter() - start
results_table["llama3.2 (Ollama)"] = {"time": llama_time, "dim": len(llama_vecs[0])}

print(f"llama3.2:       {llama_time:.2f}s   dim={len(llama_vecs[0])}")

llama3.2:       11.18s   dim=3072


In [6]:
# Model 2: nomic-embed-text via Ollama (run: ollama pull nomic-embed-text)
try:
    start = time.perf_counter()
    nomic_vecs = [embed_ollama(t, model="nomic-embed-text") for t in BENCHMARK_TEXTS]
    nomic_time = time.perf_counter() - start
    results_table["nomic-embed-text (Ollama)"] = {"time": nomic_time, "dim": len(nomic_vecs[0])}
    print(f"nomic-embed-text: {nomic_time:.2f}s   dim={len(nomic_vecs[0])}")
except Exception as e:
    print(f"nomic-embed-text not available: {e}")
    print("Run: ollama pull nomic-embed-text  to enable this model.")
    nomic_vecs = None

nomic-embed-text: 12.08s   dim=768


In [7]:
# Model 3: all-MiniLM-L6-v2 via SentenceTransformers (CPU, no Ollama needed)
start = time.perf_counter()
minilm = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
minilm_vecs = minilm.encode(BENCHMARK_TEXTS).tolist()
minilm_time = time.perf_counter() - start
results_table["all-MiniLM-L6-v2"] = {"time": minilm_time, "dim": len(minilm_vecs[0])}
print(f"all-MiniLM-L6-v2: {minilm_time:.2f}s   dim={len(minilm_vecs[0])}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

all-MiniLM-L6-v2: 90.33s   dim=384


In [8]:
print("\n=== Benchmark Summary ===")
print(f"{'Model':<35} {'Time (s)':>10} {'Dimensions':>12}")
print("-" * 60)
for model_name, info in results_table.items():
    print(f"{model_name:<35} {info['time']:>10.2f} {info['dim']:>12}")


=== Benchmark Summary ===
Model                                 Time (s)   Dimensions
------------------------------------------------------------
llama3.2 (Ollama)                        11.18         3072
nomic-embed-text (Ollama)                12.08          768
all-MiniLM-L6-v2                         90.33          384


### 3b) Quality check — compare top-1 results per model
Same query, three embedding models. Does the top hit change?

In [9]:
def top1_from_vecs(query_vec: list[float], corpus_vecs: list[list[float]], corpus_texts: list[str]) -> str:
    scores = [(cosine_similarity(query_vec, v), t) for v, t in zip(corpus_vecs, corpus_texts)]
    return max(scores, key=lambda x: x[0])

quality_query = "What database stores dense vectors for AI?"

# llama3.2 top-1
qv_llama = embed_ollama(quality_query, model="llama3.2")
score, text = top1_from_vecs(qv_llama, llama_vecs, BENCHMARK_TEXTS)
print(f"llama3.2       | score={score:.4f} | '{text}'")

# nomic top-1 (if available)
if nomic_vecs:
    qv_nomic = embed_ollama(quality_query, model="nomic-embed-text")
    score, text = top1_from_vecs(qv_nomic, nomic_vecs, BENCHMARK_TEXTS)
    print(f"nomic-embed    | score={score:.4f} | '{text}'")

# MiniLM top-1
qv_mini = minilm.encode([quality_query]).tolist()[0]
score, text = top1_from_vecs(qv_mini, minilm_vecs, BENCHMARK_TEXTS)
print(f"all-MiniLM     | score={score:.4f} | '{text}'")

llama3.2       | score=0.4235 | 'Embeddings encode semantic meaning in dense vectors.'
nomic-embed    | score=0.6977 | 'Embeddings encode semantic meaning in dense vectors.'
all-MiniLM     | score=0.5345 | 'ChromaDB is a lightweight vector database.'


---
## Part 4 — Store Vectors in ChromaDB

**ChromaDB** is an open-source, local-first vector database. It:
- Persists embeddings to disk
- Supports metadata filtering
- Works entirely offline

We'll use it with our Ollama embeddings.

In [10]:
import chromadb
from pathlib import Path

DB_PATH = str(Path("./chroma_week1").resolve())

client = chromadb.PersistentClient(path=DB_PATH)

# Delete collection if it exists from a previous run
try:
    client.delete_collection("rag_week1")
except Exception:
    pass

collection = client.create_collection(
    name="rag_week1",
    metadata={"hnsw:space": "cosine"},
)

print("ChromaDB collection created:", collection.name)
print("Persisted at:", DB_PATH)

ChromaDB collection created: rag_week1
Persisted at: C:\Users\lolen\Downloads\Jupyter Mac\Python\2-AI\LLM\LLM-Repo\Topics\AI_Eng_Path\Step_3_Retrieval_Systems_RAG\chroma_week1


### 4b) Ingest Documents — Embed + Index

In [11]:
documents = [
    {"id": "doc1", "text": "Retrieval-Augmented Generation improves LLM accuracy by grounding answers in external documents.", "category": "RAG"},
    {"id": "doc2", "text": "ChromaDB is an open-source vector database that runs locally and supports metadata filtering.", "category": "VectorDB"},
    {"id": "doc3", "text": "Cosine similarity is the standard metric for comparing embedding vectors in semantic search.", "category": "Embeddings"},
    {"id": "doc4", "text": "Large language models like llama3.2 can generate high quality text embeddings from raw input prompts.", "category": "LLM"},
    {"id": "doc5", "text": "Chunking strategies determine how documents are split before embedding. Chunk size affects retrieval quality.", "category": "RAG"},
    {"id": "doc6", "text": "Ollama lets you run LLMs locally on your machine without an API key or internet connection.", "category": "Ollama"},
    {"id": "doc7", "text": "Top-k retrieval returns the k most similar vectors to a query vector from the indexed store.", "category": "Retrieval"},
    {"id": "doc8", "text": "Semantic search uses embeddings to find relevant documents based on meaning, not just keyword overlap.", "category": "Retrieval"},
]

print("Embedding and indexing documents...")
start = time.perf_counter()

collection.add(
    ids=[d["id"] for d in documents],
    documents=[d["text"] for d in documents],
    embeddings=[embed_ollama(d["text"]) for d in documents],
    metadatas=[{"category": d["category"]} for d in documents],
)

elapsed = time.perf_counter() - start
print(f"Indexed {len(documents)} documents in {elapsed:.2f}s")
print("Collection count:", collection.count())

Embedding and indexing documents...
Indexed 8 documents in 22.81s
Collection count: 8


### 4c) Semantic Search — Query the Vector Store

In [12]:
def query_chroma(query: str, collection, top_k: int = 3, where: dict = None) -> None:
    query_vec = embed_ollama(query)

    kwargs = dict(
        query_embeddings=[query_vec],
        n_results=top_k,
        include=["documents", "distances", "metadatas"],
    )
    if where:
        kwargs["where"] = where

    results = collection.query(**kwargs)

    print(f"Query: \"{query}\"")
    if where:
        print(f"Filter: {where}")
    print()
    for rank, (doc, dist, meta) in enumerate(
        zip(results["documents"][0], results["distances"][0], results["metadatas"][0]), 1
    ):
        sim = 1 - dist
        print(f"  #{rank} [sim={sim:.4f}] [{meta['category']}] {doc}")

query_chroma("How do I store embeddings on disk?", collection, top_k=3)

Query: "How do I store embeddings on disk?"

  #1 [sim=0.3665] [Retrieval] Top-k retrieval returns the k most similar vectors to a query vector from the indexed store.
  #2 [sim=0.3651] [Embeddings] Cosine similarity is the standard metric for comparing embedding vectors in semantic search.
  #3 [sim=0.3612] [LLM] Large language models like llama3.2 can generate high quality text embeddings from raw input prompts.


In [13]:
query_chroma("What makes a LLM answer more factual?", collection, top_k=3)

Query: "What makes a LLM answer more factual?"

  #1 [sim=0.5003] [LLM] Large language models like llama3.2 can generate high quality text embeddings from raw input prompts.
  #2 [sim=0.4820] [Embeddings] Cosine similarity is the standard metric for comparing embedding vectors in semantic search.
  #3 [sim=0.4710] [RAG] Retrieval-Augmented Generation improves LLM accuracy by grounding answers in external documents.


### 4d) Metadata Filtering
ChromaDB supports filtering results by metadata fields before or after vector search.

In [14]:
# Only return docs with category == 'RAG'
query_chroma(
    query="How does retrieval improve generation?",
    collection=collection,
    top_k=2,
    where={"category": {"$eq": "RAG"}},
)

Query: "How does retrieval improve generation?"
Filter: {'category': {'$eq': 'RAG'}}

  #1 [sim=0.5799] [RAG] Retrieval-Augmented Generation improves LLM accuracy by grounding answers in external documents.
  #2 [sim=0.5299] [RAG] Chunking strategies determine how documents are split before embedding. Chunk size affects retrieval quality.


---
## Part 5 — Persist and Reload the Vector Store
Verify the index survives between sessions by reloading from disk.

In [15]:
# Re-open the client from disk (simulates a new session)
client2 = chromadb.PersistentClient(path=DB_PATH)
collection2 = client2.get_collection("rag_week1")

print("Reloaded collection from disk")
print("Document count:", collection2.count())

# Quick query to confirm data survived
query_chroma("offline local LLM inference", collection2, top_k=2)

Reloaded collection from disk
Document count: 8
Query: "offline local LLM inference"

  #1 [sim=0.4445] [LLM] Large language models like llama3.2 can generate high quality text embeddings from raw input prompts.
  #2 [sim=0.4299] [Ollama] Ollama lets you run LLMs locally on your machine without an API key or internet connection.


---
## Part 6 — Mini RAG Pipeline (Retrieve → Generate with Ollama)
Tie it all together: retrieve relevant context then generate a grounded answer using `llama3.2`.

In [16]:
def retrieve_context(query: str, collection, top_k: int = 3) -> list[str]:
    query_vec = embed_ollama(query)
    results = collection.query(
        query_embeddings=[query_vec],
        n_results=top_k,
        include=["documents"],
    )
    return results["documents"][0]


def generate_answer(query: str, context_chunks: list[str]) -> str:
    context = "\n".join(f"- {chunk}" for chunk in context_chunks)
    prompt = (
        f"Answer the question using only the provided context.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query}\n\n"
        f"Answer:"
    )
    response = requests.post(
        f"{OLLAMA_URL}/api/generate",
        json={"model": EMBED_MODEL, "prompt": prompt, "stream": False},
        timeout=120,
    )
    response.raise_for_status()
    return response.json()["response"].strip()


def rag_query(user_question: str, collection, top_k: int = 3) -> None:
    print(f"Question: {user_question}")
    print()

    chunks = retrieve_context(user_question, collection, top_k=top_k)
    print(f"Retrieved {len(chunks)} context chunks:")
    for i, c in enumerate(chunks, 1):
        print(f"  [{i}] {c}")
    print()

    answer = generate_answer(user_question, chunks)
    print("Answer:")
    print(answer)


rag_query("What is the benefit of using vector databases for AI?", collection2)

Question: What is the benefit of using vector databases for AI?

Retrieved 3 context chunks:
  [1] Semantic search uses embeddings to find relevant documents based on meaning, not just keyword overlap.
  [2] Cosine similarity is the standard metric for comparing embedding vectors in semantic search.
  [3] Large language models like llama3.2 can generate high quality text embeddings from raw input prompts.

Answer:
Vector databases provide a way to efficiently store and query dense, high-dimensional data such as text embeddings generated by large language models like llama3.2, enabling fast similarity searches that can support semantic search use cases.


In [17]:
rag_query("How does Ollama help me work without internet?", collection2)

Question: How does Ollama help me work without internet?

Retrieved 3 context chunks:
  [1] Ollama lets you run LLMs locally on your machine without an API key or internet connection.
  [2] ChromaDB is an open-source vector database that runs locally and supports metadata filtering.
  [3] Cosine similarity is the standard metric for comparing embedding vectors in semantic search.

Answer:
Ollama helps you work without internet by allowing you to run LLMs (Large Language Models) locally on your machine, effectively offline.


---
## Week 1 Summary

| Concept | What you did |
|---|---|
| Embeddings | Generated vectors with `llama3.2` via Ollama `/api/embeddings` |
| Cosine similarity | Implemented from scratch, verified on related/unrelated sentences |
| Nearest neighbour | Built brute-force similarity search |
| Model comparison | Benchmarked `llama3.2`, `nomic-embed-text`, `all-MiniLM-L6-v2` |
| ChromaDB | Created a persistent collection, ingested docs, queried with filters |
| RAG pipeline | Retrieved context + generated grounded answers with Ollama |

## Week 1 Reflection
- Which model gave the most semantically accurate top-1 result?
- How does chunk granularity affect the relevance of retrieved results?
- Where did the RAG answer diverge from what you expected?